# Fine-tuning BERTimbau v4 na T4 (Colab)

Driver do `train_bertimbau_v4.py` conforme `models/v4/spec_treino_v4.md`.
Cada celula de treino e idempotente: o script detecta `last/` e retoma com
`--resume`; nada depende de estado so em RAM. Artefatos ficam em `/content` e
no Drive.

## 1. Ambiente

In [ ]:
!nvidia-smi
import platform
import torch
import transformers

print("python", platform.python_version())
print("torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    cap = tuple(torch.cuda.get_device_capability(0))
    name = torch.cuda.get_device_name(0)
    print("GPU", name, "capability", cap)
    if cap != (7, 5):
        print("AVISO: a spec foi dimensionada para Tesla T4 (7,5); "
              "VRAM/throughput medidos aqui podem divergir da tabela do PLANO.")
    else:
        print("OK: T4 (sm_75) -> fp16 + GradScaler (bf16 nao nativo)")
else:
    print("AVISO: sem CUDA; o treino caira para CPU e nao serve para os runs cheios.")

## 2. Instalacao (nunca reinstalar torch)

In [ ]:
!pip -q install "transformers==5.17.0" scikit-learn tqdm pyarrow
import sklearn
import pyarrow
import transformers

print("transformers", transformers.__version__,
      "| sklearn", sklearn.__version__,
      "| pyarrow", pyarrow.__version__)

## 3. Drive + copia dos dados + verificacao de sha256

Copia `v4_pool.parquet`, `v4_splits.parquet`, `prepare_stats.json` e o script
para `/content/fakenewsbr/` e confere o sha256 do pool contra o
`prepare_stats.json` (aborta se divergir).

In [ ]:
import hashlib
import json
import os
import shutil

from google.colab import drive

drive.mount("/content/drive")

D = "/content/drive/MyDrive/FakenewsBR/v4"
SRC, DST = f"{D}/data", "/content/fakenewsbr"
os.makedirs(DST, exist_ok=True)
for f in ("v4_pool.parquet", "v4_splits.parquet", "prepare_stats.json",
          "train_bertimbau_v4.py"):
    shutil.copy(f"{SRC}/{f}", f"{DST}/{f}")
    print("copiado", f)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


stats = json.load(open(f"{DST}/prepare_stats.json", encoding="utf-8"))
want = stats["sha256"]["v4_pool.parquet"]
got = sha256(f"{DST}/v4_pool.parquet")
print("sha256 pool:", got)
assert got == want, f"sha256 divergente: esperado {want}, obtido {got}"
print("sha256 OK vs prepare_stats.json")

## 4. Smoke na T4 (obrigatorio antes de qualquer run cheio)

Mede tokens/s, VRAM de pico e recalcula o ETA dos runs com o valor medido.

In [ ]:
!python /content/fakenewsbr/train_bertimbau_v4.py \
  --data /content/fakenewsbr/v4_pool.parquet \
  --splits /content/fakenewsbr/v4_splits.parquet \
  --split-col bal_iid --out /content/smoke --smoke --smoke-n 512 \
  --amp auto --device auto --workers 2
import json

h = json.load(open("/content/smoke/history.json", encoding="utf-8"))
tok_s = h[-1]["tokens_per_s"]
vram = h[-1]["vram_peak_gb"]
print(f"SMOKE medido: {tok_s:.0f} tokens/s pagos | VRAM pico {vram:.2f} GB")
assert vram <= 12.0, "VRAM de pico acima de 12 GB em 32x192; revisar a escada de OOM"

tok_ep = {"R0": (1.933e6, 3), "R1": (2.1e6, 3), "A1": (3.7e6, 2), "A2": (2.213e6, 3)}
print(f"{'run':<4} {'tokens/epoca':>12} {'min/epoca':>10} {'total':>8}")
for k, (v, ep) in tok_ep.items():
    m = v / tok_s / 60
    print(f"{k:<4} {v/1e6:>10.2f}M {m:>10.1f} {ep*m:>8.1f} min")

## 5. Run default R0 (`bal_iid`, 192, freeze 6)

A celula checa se existe `last/trainer_state.json` no Drive e acrescenta
`--resume` automaticamente (idempotente).

In [ ]:
import os

D = "/content/drive/MyDrive/FakenewsBR/v4"
RUN = f"{D}/runs/bal_iid_ml192_f6_off_seed42"
RESUME_FLAG = "--resume" if os.path.exists(f"{RUN}/last/trainer_state.json") else ""
print("retomando run existente no Drive" if RESUME_FLAG else "run novo")
!python /content/fakenewsbr/train_bertimbau_v4.py \
  --data /content/fakenewsbr/v4_pool.parquet \
  --splits /content/fakenewsbr/v4_splits.parquet \
  --split-col bal_iid --eval-col bal_iid --eval-value test \
  --out /content/runs/bal_iid_ml192_f6_off_seed42 \
  --drive-out $RUN $RESUME_FLAG \
  --max-length 192 --batch-size 32 --epochs 3 --patience 1 \
  --lr 2e-5 --freeze-layers 6 --seed 42 --amp auto --workers 2

## 6. OOD WhatsApp R1 (`bal_ood_wa`)

In [ ]:
import os

D = "/content/drive/MyDrive/FakenewsBR/v4"
RUN = f"{D}/runs/bal_ood_wa_ml192_f6_off_seed42"
RESUME_FLAG = "--resume" if os.path.exists(f"{RUN}/last/trainer_state.json") else ""
print("retomando run existente no Drive" if RESUME_FLAG else "run novo")
!python /content/fakenewsbr/train_bertimbau_v4.py \
  --data /content/fakenewsbr/v4_pool.parquet \
  --splits /content/fakenewsbr/v4_splits.parquet \
  --split-col bal_ood_wa --eval-col bal_ood_wa --eval-value test \
  --out /content/runs/bal_ood_wa_ml192_f6_off_seed42 \
  --drive-out $RUN $RESUME_FLAG \
  --max-length 192 --batch-size 32 --epochs 3 --patience 1 \
  --lr 2e-5 --freeze-layers 6 --seed 42 --amp auto --workers 2

## 7. Ablacoes A1/A2/A3 (executar uma por vez)

Defina `RUN` como `"A1"`, `"A2"` ou `"A3"`. Cada uma e idempotente com
`--resume` e espelha no Drive.

In [ ]:
import os

D = "/content/drive/MyDrive/FakenewsBR/v4"
RUN = ""  # "A1" | "A2" | "A3"

if RUN == "A1":
    # A1: pool completo + pesos DFR (2 epocas)
    dst = f"{D}/runs/full_iid_dfr"
    RESUME_FLAG = "--resume" if os.path.exists(f"{dst}/last/trainer_state.json") else ""
    !python /content/fakenewsbr/train_bertimbau_v4.py \
      --data /content/fakenewsbr/v4_pool.parquet \
      --splits /content/fakenewsbr/v4_splits.parquet \
      --ablate fullpool --out /content/runs/full_iid_dfr \
      --drive-out $dst $RESUME_FLAG
elif RUN == "A2":
    # A2: max_length 256
    dst = f"{D}/runs/bal_iid_ml256"
    RESUME_FLAG = "--resume" if os.path.exists(f"{dst}/last/trainer_state.json") else ""
    !python /content/fakenewsbr/train_bertimbau_v4.py \
      --data /content/fakenewsbr/v4_pool.parquet \
      --splits /content/fakenewsbr/v4_splits.parquet \
      --ablate maxlen256 --out /content/runs/bal_iid_ml256 \
      --drive-out $dst $RESUME_FLAG
elif RUN == "A3":
    # A3: DFR nos informativos
    dst = f"{D}/runs/bal_iid_dfr"
    RESUME_FLAG = "--resume" if os.path.exists(f"{dst}/last/trainer_state.json") else ""
    !python /content/fakenewsbr/train_bertimbau_v4.py \
      --data /content/fakenewsbr/v4_pool.parquet \
      --splits /content/fakenewsbr/v4_splits.parquet \
      --ablate dfr --out /content/runs/bal_iid_dfr \
      --drive-out $dst $RESUME_FLAG
else:
    print("defina RUN = 'A1' | 'A2' | 'A3'")

## 8. Coleta: resumo dos runs + copia para o Drive

In [ ]:
import glob
import json
import os
import shutil

import pandas as pd

D = "/content/drive/MyDrive/FakenewsBR/v4"
rows = []
for p in sorted(glob.glob(f"{D}/runs/*/metrics.json")):
    m = json.load(open(p, encoding="utf-8"))
    t = m.get("test") or {}
    o = m.get("ood") or {}
    rows.append({
        "run": m.get("run_id"), "n": t.get("n"), "acc": t.get("acc"),
        "macro_f1": t.get("macro_f1"),
        "worst_group_macro_f1": t.get("worst_group_macro_f1"),
        "ece": t.get("ece"), "brier": t.get("brier"),
        "ood_macro_f1": o.get("macro_f1"),
        "best_epoch": m.get("best_epoch"),
    })
resumo = pd.DataFrame(rows)
pd.set_option("display.width", 220)
print(resumo.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
resumo.to_csv(f"{D}/resumo.csv", index=False)

for src in glob.glob("/content/runs/*"):
    dst = f"{D}/runs/{os.path.basename(src)}"
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
print("copiado /content/runs ->", f"{D}/runs")

## 9. Download: zip de `best/` + JSON/CSVs (sem `last/`)

In [ ]:
import glob
import os
import zipfile

D = "/content/drive/MyDrive/FakenewsBR/v4"
KEEP = ["metrics.json", "calibration.json", "predictions.csv", "per_group.csv",
        "reliability.csv", "history.json", "run_config.json", "token_stats.json"]
zip_path = "/content/fakenewsbr_v4_artefatos.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for run in sorted(glob.glob(f"{D}/runs/*/")):
        rid = os.path.basename(os.path.normpath(run))
        for f in KEEP:
            p = os.path.join(run, f)
            if os.path.exists(p):
                z.write(p, f"{rid}/{f}")
        best = os.path.join(run, "best")
        if os.path.isdir(best):
            for root, _, files in os.walk(best):
                for f in files:
                    p = os.path.join(root, f)
                    z.write(p, f"{rid}/best/{os.path.relpath(p, best)}")
print("zip:", zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("download manual:", zip_path, "|", e)